# 第14周 · 预训练模型微调与 LoRA（含数学工作坊 3：概率与 KL）

**本周结束时你能做到**：用 Hugging Face transformers 加载 DistilBERT，自己写训练循环在 IMDB 上微调它（也认识 Trainer 的写法）；区分线性探针、部分冻结、全量微调与 LoRA，并会数每种方案的可训练参数；从零写出一个 LoRA 线性层，验证它"一开始不改变模型"；用代码验证下周 VAE 要用的几条概率公式——期望、似然、交叉熵 = 负对数似然、熵、KL 散度。

上周的线性探针没有赢过 TF-IDF（DistilBERT 均值池化嵌入 + 逻辑回归 0.8625，TF-IDF 0.8665）：冻结的通用表示，配上一个线性分类器，不够。今天的问题是：**让预训练模型自己的权重也跟着任务动起来，会怎样？要动多少参数才够？**

本周蓝本：

- 微调：Hugging Face transformers 官方教程 [Text classification](https://huggingface.co/docs/transformers/tasks/sequence_classification)（transformers 仓库，Apache-2.0）的流程：`AutoTokenizer` → `AutoModelForSequenceClassification(num_labels=2)` → 训练。课程以**自己写的训练循环**为主（第 8 周的 `train_one_epoch` 几乎原样搬来），Trainer 只演示一格。
- LoRA：Hu et al. 2021, *LoRA: Low-Rank Adaptation of Large Language Models*（[arXiv:2106.09685](https://arxiv.org/abs/2106.09685)）；从零实现的写法参考 Sebastian Raschka 的博文 [Improving LoRA: Implementing DoRA from Scratch](https://magazine.sebastianraschka.com/p/lora-and-dora-from-scratch)（2024）；库用 Hugging Face [PEFT](https://github.com/huggingface/peft)（Apache-2.0）。
- 数学工作坊：d2l.ai 附录 [Information Theory](https://d2l.ai/chapter_appendix-mathematics-for-deep-learning/information-theory.html)（熵、KL、交叉熵），以及 Kingma & Welling 2013（[arXiv:1312.6114](https://arxiv.org/abs/1312.6114)）附录 B 的高斯 KL 闭式解。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")
# 第 14 周追加：Hugging Face transformers（5.x 写法）与 PEFT。Colab 预装了 transformers，一般只需装 peft
for pkg, spec in (("transformers", "transformers>=5.22,<6"), ("peft", "peft>=0.21,<1")):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", spec], check=True)
print("transformers / peft 已就绪")

In [ ]:
import os, time, math, tempfile
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")   # 关掉 tokenizers 库的多进程提示
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from zero2researcher import data, experiments
from zero2researcher.checks import week14 as ck

# 本周的 device 选择跳过 MPS：transformers 在 Apple 芯片的 MPS 上训练时会碰到不支持的算子
# （注意力里的 dropout），所以只在 CUDA（Colab GPU）与 CPU 之间选
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transformers.logging.disable_progress_bar()
print("torch", torch.__version__, "| transformers", transformers.__version__, "| device:", device)

---
## S1 · 微调 DistilBERT

**DistilBERT**（Sanh et al. 2019, [arXiv:1910.01108](https://arxiv.org/abs/1910.01108)）是 BERT 的"蒸馏"版：用一个小模型去模仿大模型的输出。原文摘要的说法是：把 BERT 缩小 40%，保留 97% 的语言理解能力，快 60%。它是 6 层、隐藏维度 768 的 Transformer 编码器——和你第 11 周从零写的模型是同一种东西，只是更大，并且已经在大量英文文本上用**掩码语言模型**目标（上周 S3）预训练过。

### 1. 分词器：子词，而不是整词

第 10–11 周我们用工具包里的 `text.tokenize` 按整词切分、自己建词表，词表外的词一律变成 `<unk>`。预训练模型自带分词器，**必须用它**——模型只认识它预训练时见过的那套 token id。DistilBERT 用的是 WordPiece：常见词是一个 token，罕见词被拆成几个子词（`##` 表示"接在前一个子词后面"），所以几乎不会出现 `<unk>`。

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
REVISION = "12040accade4e8a0f71eabdb258fecc2e7e948be"   # 固定模型版本（Hub 上的提交号）：模型仓库以后更新，课程里的数字也能复现
tok = AutoTokenizer.from_pretrained(MODEL_NAME, revision=REVISION)   # 首次运行会下载（连同模型约 270MB），之后读缓存
print("词表大小:", tok.vocab_size)

enc = tok("This movie was not good at all!")
print("input_ids:", enc["input_ids"])
print("tokens:   ", tok.convert_ids_to_tokens(enc["input_ids"]))
print("罕见词被拆成子词:", tok.tokenize("Unbelievably overacted"))

两个特殊 token：开头的 `[CLS]`（id 101）和结尾的 `[SEP]`（id 102）是分词器自动加的，预训练时就是这个格式。

模型一次吃一个**矩形**的 batch，所以要把长短不一的影评整理成同样长度：

- `truncation=True, max_length=128`：超过 128 个 token 的截掉后半段；
- `padding="max_length"`：不足 128 的在后面补 `[PAD]`（id 0）；
- `attention_mask`：真实 token 为 1、补出来的 `[PAD]` 为 0——注意力会忽略 mask 为 0 的位置（第 11 周你亲手写过的 padding mask，这里分词器替你生成）。

本周 CPU 上统一用一个**缩小配置**：训练集取课程 IMDB 子集的前 500 条、测试集取前 500 条（两者都正负各半）、最长 128 个 token。

In [ ]:
imdb_train, imdb_test = data.load_imdb("train"), data.load_imdb("test")
N_TRAIN, N_TEST, MAX_LEN = 500, 500, 128            # CPU 缩小配置；Colab T4 的大配置见本节末尾
train_df, test_df = imdb_train.iloc[:N_TRAIN], imdb_test.iloc[:N_TEST]
print(f"训练 {len(train_df)} 条（正例比例 {train_df['label'].mean():.3f}）| 测试 {len(test_df)} 条（正例比例 {test_df['label'].mean():.3f}）")


def encode(texts):
    # 一组影评 → (input_ids, attention_mask)，都是 (N, MAX_LEN) 的整数张量
    enc = tok(list(texts), truncation=True, padding="max_length", max_length=MAX_LEN, return_tensors="pt")
    return enc["input_ids"], enc["attention_mask"]


ids_tr, mask_tr = encode(train_df["text"])
ids_te, mask_te = encode(test_df["text"])
y_tr = torch.tensor(train_df["label"].to_numpy())
y_te = torch.tensor(test_df["label"].to_numpy())
print("ids_tr:", tuple(ids_tr.shape), "| 被截断的影评比例:", round((mask_tr[:, -1] == 1).float().mean().item(), 3))

大约九成影评超过 128 个 token（这 500 条的中位数约 234 个），都被截掉了后半段——这是缩小配置的第一个代价（Colab 配置用 256）。

### 2. 模型：预训练的身体 + 一个新的分类头

`AutoModelForSequenceClassification` 做了两件事：加载预训练好的 DistilBERT 身体，再在上面接一个**随机初始化**的分类头（`num_labels=2`）。加载时 transformers 会打印一份报告，读懂它：

In [ ]:
experiments.set_seed(0)            # 分类头是随机初始化的：先固定种子
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, revision=REVISION, num_labels=2)

- **MISSING**：`pre_classifier`、`classifier` 两层在预训练权重里没有——它们是新建的、随机初始化的，**必须训练**；
- **UNEXPECTED**：`vocab_*` 几层是预训练时做"完形填空"（预测被遮住的词）用的输出头，分类任务用不上，被丢掉了。

后面还要加载好几次模型，把这份报告关掉；然后数一数参数：

In [ ]:
transformers.logging.set_verbosity_error()      # 之后加载模型不再打印报告

body = sum(p.numel() for p in model.distilbert.parameters())
head = sum(p.numel() for name, p in model.named_parameters() if not name.startswith("distilbert"))
print(f"DistilBERT 身体 {body:,}（≈{body / 1e6:.1f}M）| 新分类头 {head:,} | 合计 {body + head:,}")
print(model.pre_classifier, model.classifier, sep="\n")

分类头是 `pre_classifier`（768→768，ReLU）+ `classifier`（768→2），读的是 `[CLS]` 位置的最后一层表示。

**先做第 9 周的体检 ①**：随机初始化的头还"什么都不知道"，2 类交叉熵的初始损失应该约等于 $\ln 2 \approx 0.693$。transformers 的模型在你传入 `labels` 时会顺手算好交叉熵损失：

In [ ]:
model.to(device).eval()
with torch.no_grad():
    out = model(input_ids=ids_tr[:32].to(device), attention_mask=mask_tr[:32].to(device), labels=y_tr[:32].to(device))
print("logits 的形状:", tuple(out.logits.shape), "| 前 3 行:", np.round(out.logits[:3].cpu().numpy(), 3).tolist())
print(f"初始损失 {out.loss.item():.4f}   理论值 ln(2) = {np.log(2):.4f}")

### 3. 自己写训练循环

和第 8 周相比只有两处变化：一个 batch 是 `(input_ids, attention_mask, label)` 三样东西；损失由模型自己算（`out.loss`），不必再写 `criterion`。五步核心一模一样。

优化器换成 **AdamW**（Loshchilov & Hutter 2019）：Adam + 解耦的权重衰减，是微调 Transformer 的标准选择。学习率是 **2e-5**——比第 9 周的 1e-3 小 50 倍。预训练权重已经很好了，微调只需要轻轻推一下；步子太大会把预训练学到的东西冲掉。

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    # 第 8 周的 train_one_epoch：batch 换成 (ids, mask, y) 三样，损失由模型自己算
    model.train()
    total, n = 0.0, 0
    for ids, mask, yb in loader:
        ids, mask, yb = ids.to(device), mask.to(device), yb.to(device)
        out = model(input_ids=ids, attention_mask=mask, labels=yb)
        optimizer.zero_grad()
        out.loss.backward()
        optimizer.step()
        total += out.loss.item() * len(yb)
        n += len(yb)
    return total / n


def evaluate(model, ids, mask, y, batch_size=64):
    # 分批前向（一次喂 500 条会占很多内存），返回准确率
    model.eval()
    correct = 0
    with torch.no_grad():
        for i in range(0, len(y), batch_size):
            logits = model(input_ids=ids[i:i + batch_size].to(device),
                           attention_mask=mask[i:i + batch_size].to(device)).logits
            correct += (logits.argmax(dim=1).cpu() == y[i:i + batch_size]).sum().item()
    return correct / len(y)

训练 3 个 epoch（每个 epoch 32 步），每个 epoch 结束时在测试集上量一次。CPU 上大约两分钟——趁它跑的时候想一想：只训练 1 个 epoch 会怎样？

In [ ]:
EPOCHS, LR, BATCH = 3, 2e-5, 16
experiments.set_seed(0)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
loader = DataLoader(TensorDataset(ids_tr, mask_tr, y_tr), batch_size=BATCH, shuffle=True)
history = []
t0 = time.time()
for epoch in range(EPOCHS):
    loss = train_one_epoch(model, loader, optimizer, device)
    acc = evaluate(model, ids_te, mask_te, y_te)
    history.append((loss, acc))
    print(f"epoch {epoch + 1}: 训练损失 {loss:.3f} | 测试准确率 {acc:.3f} | 累计 {time.time() - t0:.0f} 秒")
ft_acc, ft_seconds = history[-1][1], time.time() - t0
ck.finetune_accuracy(ft_acc)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(range(1, EPOCHS + 1), [a for _, a in history], "o-", label="DistilBERT fine-tuned (500 train)")
ax.axhline(0.8665, color="gray", ls="--", label="TF-IDF + LR, 5000 train (week 6)")
ax.axhline(0.5, color="lightgray", ls=":", label="Chance")
ax.set_xticks(range(1, EPOCHS + 1))
ax.set_xlabel("Epoch")
ax.set_ylabel("Test accuracy (first 500 reviews)")
ax.set_ylim(0.45, 0.95)
ax.set_title("Fine-tuning learns within a few epochs, but 500 reviews stay below TF-IDF")
ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.show()

### 4. 放进历届 IMDB 成绩单

TF-IDF 基线 0.8665 是用 **5000 条**训练出来的，而我们的 DistilBERT 只看了 500 条、每条只读前 128 个 token——这样比不公平。补两行"同口径"的 TF-IDF：同一份测试集（前 500 条），以及同样只用 500 条训练：

In [ ]:
def tfidf_accuracy(train_texts, train_labels, test_texts, test_labels):
    # 第 6 周的基线：TfidfVectorizer(min_df=2) + LogisticRegression(max_iter=1000)，词表只在训练集上拟合
    vec = TfidfVectorizer(min_df=2)
    clf = LogisticRegression(max_iter=1000).fit(vec.fit_transform(train_texts), train_labels)
    return clf.score(vec.transform(test_texts), test_labels)


tfidf_full = tfidf_accuracy(imdb_train["text"], imdb_train["label"], imdb_test["text"], imdb_test["label"])
tfidf_full_500 = tfidf_accuracy(imdb_train["text"], imdb_train["label"], test_df["text"], test_df["label"])
tfidf_same = tfidf_accuracy(train_df["text"], train_df["label"], test_df["text"], test_df["label"])

board = pd.DataFrame([
    ("TF-IDF + LR", 5000, "全部 2000", tfidf_full, "第 6 周基线（本格重算）"),
    ("TF-IDF + LR", 5000, "前 500", tfidf_full_500, "本周：同一份测试集"),
    ("TF-IDF + LR", 500, "前 500", tfidf_same, "本周：同样只用 500 条训练"),
    ("TextCNN（从零训练）", 5000, "全部 2000", 0.77, "第 10 周，约（5 epochs）"),
    ("小 Transformer（从零训练）", 5000, "全部 2000", 0.81, "第 11 周，约 0.80–0.82"),
    ("DistilBERT 均值池化 + LR（线性探针）", 5000, "全部 2000", 0.8625, "第 13 周"),
    ("DistilBERT 全量微调", N_TRAIN, "前 500", ft_acc, f"本周，CPU {ft_seconds / 60:.1f} 分钟"),
], columns=["model", "n_train", "test", "test_acc", "note"])
board.round(4)

读这张表要分两层：

1. **同样只给 500 条训练数据**时，微调过的 DistilBERT 比 TF-IDF 高出几个百分点——预训练带来的"语言常识"在小数据下最值钱，这是迁移学习的核心卖点（只有 1 个种子，作业里要用 3 个种子确认）；
2. 但它还没赢过**用 5000 条训练**的 TF-IDF。这不是方法失败，而是我们把数据（500 条）、长度（128 token）和训练量（96 步）都缩小到了 CPU 跑得动的程度。公开的参考结果：DistilBERT 原文（Sanh et al. 2019）表 2 报告，在完整 IMDb（25,000 条训练）上微调后测试准确率为 92.82%（BERT-base 93.46%）。

**请在 Colab T4 上跑大配置**：把上面的 `N_TRAIN, MAX_LEN` 改成 `5000, 256`、测试集用全部 2000 条，2–3 个 epoch，并开混合精度（fp16，作业里有现成写法）——这才是"微调赢过基线"该出现的地方。本周作业的研究实验就在这个配置上做。

### 5. 冻结一部分层

全量微调要更新 6700 万个参数；另一个极端是上周的线性探针——身体完全冻结，只训练最后的分类器。中间有一整条光谱：**冻结底部的若干层，只训练顶部**。冻结的写法就是第 8 周见过的 `requires_grad = False`——冻结的参数不再计算梯度，优化器也就不会动它们：

In [ ]:
def freeze_bottom(model, n_layers):
    # 冻结 embeddings 和最底下 n_layers 个 Transformer 层；其余（顶层 + 分类头）保持可训练
    for p in model.parameters():
        p.requires_grad = True
    for p in model.distilbert.embeddings.parameters():
        p.requires_grad = False
    for layer in model.distilbert.transformer.layer[:n_layers]:
        for p in layer.parameters():
            p.requires_grad = False


def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


demo = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, revision=REVISION, num_labels=2)
total = sum(p.numel() for p in demo.parameters())
for n_layers in (0, 2, 4, 6):
    freeze_bottom(demo, n_layers)
    n = count_trainable(demo)
    print(f"冻结 embeddings + 底部 {n_layers} 层：可训练 {n:>11,}（{100 * n / total:5.1f}%）")
ck.trainable_params(count_trainable(demo), demo)

最后一行（6 层全冻）就是"只训分类头"——线性探针的近亲（头是两层的 MLP，而且身体在训练时仍是 train 模式）。注意 embeddings 占了约 2380 万参数（其中词表 30,522 个词 × 768 维 ≈ 2344 万），每个 Transformer 层约 709 万。

| 方案 | 可训练参数 | 训练时要做什么 |
| --- | --- | --- |
| 线性探针 / 只训分类头 | ≈0.59M | 身体只前向；可以把特征一次算好缓存起来 |
| 冻结底部若干层 | 介于两者之间 | 反向传播只走到最低的未冻结层 |
| 全量微调 | ≈67M | 每个参数都有梯度和优化器状态（AdamW 每个参数再存 2 个数） |
| LoRA（下一节） | 不到 1% | 身体全冻，每层旁边挂一对小矩阵 |

### 6. Trainer：同一件事的高层写法（演示）

HF 官方教程用的是 `Trainer`：把训练循环、评估、日志、混合精度、存档都包好了。下面只走 8 步，看清楚接口长什么样即可（完整训练把 `max_steps=8` 换成 `num_train_epochs=3`）：

In [ ]:
from transformers import Trainer, TrainingArguments


class IMDBDataset(torch.utils.data.Dataset):
    # Trainer 要的数据格式：每条样本是一个 dict，键名和模型 forward 的参数名一致
    def __init__(self, ids, mask, y):
        self.ids, self.mask, self.y = ids, mask, y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return {"input_ids": self.ids[i], "attention_mask": self.mask[i], "labels": self.y[i]}


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return {"accuracy": float((logits.argmax(axis=-1) == labels).mean())}


experiments.set_seed(0)
trainer_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, revision=REVISION, num_labels=2)
args = TrainingArguments(
    output_dir=tempfile.mkdtemp(),
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    max_steps=8,                     # 演示用；完整训练写 num_train_epochs=3
    eval_strategy="no",              # 5.x 写法（4.x 叫 evaluation_strategy）；想每个 epoch 评估一次写 "epoch"
    save_strategy="no",
    logging_steps=4,
    report_to="none",
    use_cpu=not torch.cuda.is_available(),
)
trainer = Trainer(model=trainer_model, args=args,
                  train_dataset=IMDBDataset(ids_tr, mask_tr, y_tr),
                  eval_dataset=IMDBDataset(ids_te[:64], mask_te[:64], y_te[:64]),
                  processing_class=tok,      # 5.x 写法（4.x 叫 tokenizer=）
                  compute_metrics=compute_metrics)
trainer.train()
print(trainer.evaluate())

Trainer 很方便，但它是个"黑箱"：五步核心、`model.train()/eval()`、梯度清零都藏在里面。transformers 正处在 5.x 大版本的快速演进期，**网上大多数教程还是 4.x 写法**（`evaluation_strategy=`、`tokenizer=`），照抄会直接报错。所以本课程以自己写的循环为主：你清楚每一步在做什么，也不受库版本变化的影响。

---
## S2 · LoRA：只训练一个低秩的"增量"

### 思想

微调就是把每个权重矩阵从 $W_0$ 改成 $W_0 + \Delta W$。全量微调直接训练整个 $\Delta W$（一个 768×768 的矩阵就有 589,824 个数）。LoRA（Hu et al. 2021）的假设是：**适配一个新任务所需的改动 $\Delta W$ 是"低秩"的**——可以写成两个瘦长矩阵的乘积：

$$h = W_0 x + \frac{\alpha}{r}\, B A x, \qquad B \in \mathbb{R}^{d_{\text{out}} \times r},\quad A \in \mathbb{R}^{r \times d_{\text{in}}},\quad r \ll \min(d_{\text{in}}, d_{\text{out}})$$

- $W_0$ **冻结**，只训练 $A$ 和 $B$：参数量从 $d_{\text{out}} d_{\text{in}}$ 降到 $r(d_{\text{in}} + d_{\text{out}})$。768×768、$r=4$ 时是 6,144 个——约为原来的 1%；
- **初始化**（论文 §4.1）：$A$ 用随机高斯，$B$ 置零。于是一开始 $BA = 0$、$h = W_0 x$：**模型一开始就是预训练模型本身**，训练从一个好起点出发；
- 为什么不两个都置零？$B$ 的梯度是 $\delta\,(Ax)^\top$、$A$ 的梯度是 $B^\top \delta\, x^\top$（$\delta$ 是输出端传回来的梯度）——两个都是 0 时梯度也全是 0，永远学不动。一个随机、一个为零，正好既"不改变起点"又"能学"；
- $\alpha / r$ 是缩放：论文把 $\alpha$ 定为第一次尝试的 $r$ 之后就不再调——$\alpha$ 是与 $r$ 无关的常数，所以 $r$ 变大时缩放 $\alpha/r$ 变小；论文说这种缩放能减少换 $r$ 时重调超参数（如学习率）的需要；
- 训练完可以把 $\frac{\alpha}{r}BA$ 直接加回 $W_0$，推理时**没有任何额外开销**。论文报告：对 GPT-3 175B，可训练参数减少一万倍、显存减少三倍。

### 从零实现：约 20 行

照论文的记号写。Raschka 的博文结构相同，但有两处不同：$A$、$B$ 转置存放；缩放直接乘 $\alpha$、不除以 $r$（等价于换了一种 $\alpha$ 的取值约定）。我们和论文、PEFT 一致，用 $\alpha/r$：

In [ ]:
class LoRALinear(nn.Module):
    # 包住一个冻结的 nn.Linear：h = base(x) + (alpha / r) · x Aᵀ Bᵀ
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False                     # W0（和 bias）冻结
        self.scaling = alpha / r
        self.A = nn.Parameter(torch.randn(r, base.in_features) / math.sqrt(r))   # 高斯初始化
        self.B = nn.Parameter(torch.zeros(base.out_features, r))                  # 零初始化

    def forward(self, x):
        return self.base(x) + (x @ self.A.T @ self.B.T) * self.scaling


**数值验证"初始输出不变"**：包住一个 768×768 的线性层，同一批输入，比较前后输出的最大差：

In [ ]:
experiments.set_seed(0)
base = nn.Linear(768, 768)
x = torch.randn(4, 10, 768)                        # (batch, 序列长度, 768)：DistilBERT 里的形状
with torch.no_grad():
    y_base = base(x)
lora = LoRALinear(base, r=4, alpha=8)
with torch.no_grad():
    y_lora = lora(x)
n_lora = sum(p.numel() for p in lora.parameters() if p.requires_grad)
print("初始时与原层输出的最大差:", (y_lora - y_base).abs().max().item())
print(f"可训练参数 {n_lora:,}，原层 {sum(p.numel() for p in base.parameters()):,}（{100 * n_lora / 590592:.2f}%）")

with torch.no_grad():                               # 假装训练了几步：B 不再是 0
    lora.B.normal_(0, 0.01)
    y_trained = lora(x)
    merged = base.weight + lora.scaling * lora.B @ lora.A          # 把增量合并回 W0
    y_merged = x @ merged.T + base.bias
print("B 非零后输出改变了:", (y_trained - y_base).abs().max().item() > 0,
      "| 合并进 W0 后与 LoRA 层的最大差:", (y_merged - y_trained).abs().max().item())
ck.lora_init_identity(LoRALinear)

### 装进 DistilBERT：手动遍历模块

论文主实验只改注意力里的 $W_q$ 和 $W_v$。DistilBERT 每个 Transformer 层的注意力模块里，这两层叫 `q_lin` 和 `v_lin`（`print(model.distilbert.transformer.layer[0])` 可以看到 `q_lin / k_lin / v_lin / out_lin`）。把 6 层的这两个 `nn.Linear` 换成 `LoRALinear`，其余全部冻结——**只有新分类头例外**，它是随机初始化的，必须训练：

In [ ]:
def add_lora(model, r, alpha, targets=("q_lin", "v_lin")):
    for p in model.parameters():
        p.requires_grad = False
    for layer in model.distilbert.transformer.layer:
        for name in targets:
            setattr(layer.attention, name, LoRALinear(getattr(layer.attention, name), r, alpha))
    for p in list(model.pre_classifier.parameters()) + list(model.classifier.parameters()):
        p.requires_grad = True
    return model


experiments.set_seed(0)
lora_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, revision=REVISION, num_labels=2).eval()
with torch.no_grad():
    before = lora_model(input_ids=ids_te[:8], attention_mask=mask_te[:8]).logits
add_lora(lora_model, r=4, alpha=8)
with torch.no_grad():
    after = lora_model(input_ids=ids_te[:8], attention_mask=mask_te[:8]).logits
print("装上 LoRA 前后 logits 的最大差:", (after - before).abs().max().item())

ab = sum(p.numel() for name, p in lora_model.named_parameters() if p.requires_grad and name.endswith((".A", ".B")))
head_n = sum(p.numel() for name, p in lora_model.named_parameters()
             if p.requires_grad and not name.endswith((".A", ".B")))
print(f"A/B 矩阵 {ab:,} = 6 层 × 2 个模块 × r·(768 + 768) = {6 * 2 * 4 * (768 + 768):,}")
print(f"分类头 {head_n:,} | 合计可训练 {ab + head_n:,}（全模型的 {100 * (ab + head_n) / total:.2f}%）")

### 换成 PEFT 库，并对账

实际项目里不会手写替换逻辑，而是用 PEFT：告诉它秩、缩放和目标模块名，它负责替换与冻结。`task_type=TaskType.SEQ_CLS` 表示"序列分类"——PEFT 会**自动把分类头也设为可训练**（放进 `modules_to_save`），和我们手写的 `add_lora` 一样：

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model

experiments.set_seed(0)
config = LoraConfig(task_type=TaskType.SEQ_CLS, r=4, lora_alpha=8, lora_dropout=0.1,
                    target_modules=["q_lin", "v_lin"])
peft_model = get_peft_model(
    AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, revision=REVISION, num_labels=2), config)
peft_model.print_trainable_parameters()

trainable = {name: p.numel() for name, p in peft_model.named_parameters() if p.requires_grad}
peft_ab = sum(n for name, n in trainable.items() if "lora_" in name)
peft_head = sum(n for name, n in trainable.items() if "lora_" not in name)
print(f"PEFT 拆账：LoRA A/B {peft_ab:,} + 分类头 {peft_head:,} = {peft_ab + peft_head:,}")
print("和手写版一致:", (peft_ab, peft_head) == (ab, head_n))
ck.peft_accounting(sum(trainable.values()), peft_head, peft_ab, peft_model)   # 自查会拿模型独立重数一遍

`print_trainable_parameters` 报的 665,858 里，**真正属于 LoRA 的只有 73,728 个**，其余 592,130 个是分类头。只看那一行输出，很容易以为"r=4 的 LoRA 有 66 万参数"——数参数时一定要拆开看。作业的研究实验会把这笔账算到 r=16，并加上训练时间和准确率。

（PEFT 的 `lora_dropout` 是在 LoRA 支路的输入上加 dropout；它的 `lora_A` 默认用 Kaiming-uniform 而不是高斯初始化——细节不同，"$B$ 零初始化、起点不变"的核心一样。）

---
## S3 · 数学工作坊 3：概率与 KL 散度

这一节是下周 VAE 的直接前置：VAE 的损失函数 = 一项**对数似然**（重建）+ 一项 **KL 散度**。规则和前两次工作坊一样：**每条公式都用代码验证一遍**。

### 1. 三个分布

| 分布 | 描述 | 参数 | 均值 | 方差 |
| --- | --- | --- | --- | --- |
| 伯努利 $\mathrm{Bern}(p)$ | 一次抛硬币：$P(x{=}1) = p$ | $p$ | $p$ | $p(1-p)$ |
| 类别 $\mathrm{Cat}(\pi_1, \dots, \pi_K)$ | 掷一个 $K$ 面骰子：$P(x{=}k) = \pi_k$ | $\pi$（和为 1） | — | — |
| 高斯 $\mathcal{N}(\mu, \sigma^2)$ | 连续值，密度 $p(x) = \frac{1}{\sqrt{2\pi\sigma^2}} e^{-\frac{(x-\mu)^2}{2\sigma^2}}$ | $\mu, \sigma$ | $\mu$ | $\sigma^2$ |

你早就见过它们：二分类模型的 sigmoid 输出是伯努利的 $p$；十分类模型的 softmax 输出是类别分布的 $\pi$；第 9 周的 Kaiming 初始化从高斯里抽权重。抽样，看频率是否贴近概率：

In [ ]:
rng = np.random.default_rng(0)
n = 100_000

coin = rng.binomial(1, 0.3, size=n)                                  # 伯努利 p = 0.3
die = rng.choice(4, p=[0.1, 0.2, 0.3, 0.4], size=n)                  # 类别分布，4 个类
print(f"伯努利：样本均值 {coin.mean():.4f}（理论 0.3）| 样本方差 {coin.var():.4f}（理论 {0.3 * 0.7:.4f}）")
print("类别分布：各类频率", np.bincount(die, minlength=4) / n, "（理论 [0.1 0.2 0.3 0.4]）")


def gauss_pdf(x, mu, sigma):
    return np.exp(-(x - mu) ** 2 / (2 * sigma ** 2)) / np.sqrt(2 * np.pi * sigma ** 2)


g = rng.normal(1.0, 2.0, size=n)
xs = np.linspace(-7, 9, 300)
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(g, bins=80, density=True, alpha=0.5, label="100,000 samples")
ax.plot(xs, gauss_pdf(xs, 1.0, 2.0), "r", label="Density formula N(1, 2²)")
ax.set_xlabel("x")
ax.set_ylabel("Density")
ax.set_title("Samples from N(1, 2²) match the density formula")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### 2. 蒙特卡洛：用样本平均估计期望

期望是"按概率加权的平均"：$\mathbb{E}_{p}[f(x)] = \int f(x)\,p(x)\,dx$。积分很多时候算不出来，但**抽样很容易**：

$$\mathbb{E}_{p}[f(x)] \approx \frac{1}{n}\sum_{i=1}^{n} f(x_i), \qquad x_i \sim p$$

这就是蒙特卡洛估计。验证一个能算出解析解的例子：$x \sim \mathcal{N}(1, 2^2)$ 时 $\mathbb{E}[x^2] = \mu^2 + \sigma^2 = 5$。样本越多越准——误差按 $1/\sqrt{n}$ 缩小，和第 9 周 mini-batch 梯度噪声的规律完全一样（mini-batch 梯度本身就是全数据梯度的蒙特卡洛估计）：

In [ ]:
ns = [10, 100, 1_000, 10_000, 100_000, 1_000_000]
errors = []
for n_samples in ns:
    estimates = [np.mean(rng.normal(1.0, 2.0, size=n_samples) ** 2) for _ in range(20)]
    errors.append(np.mean(np.abs(np.array(estimates) - 5.0)))
    print(f"n = {n_samples:>9,}: 一次估计 {estimates[0]:.4f} | 20 次的平均绝对误差 {errors[-1]:.4f}")

fig, ax = plt.subplots(figsize=(5, 3.2))
ax.loglog(ns, errors, "o-", label="Measured")
ax.loglog(ns, errors[0] * np.sqrt(ns[0] / np.array(ns)), "--", color="gray", label="1/sqrt(n) reference")
ax.set_xlabel("Number of samples n")
ax.set_ylabel("Mean absolute error")
ax.set_title("Monte Carlo error shrinks like 1/sqrt(n)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### 3. 似然与对数似然

换个方向：**分布的参数未知，数据已知**。一枚硬币抛 10 次得到 `1 1 0 1 1 1 0 1 0 1`（7 次正面），$p$ 是多少？

把数据的概率看成参数的函数，就是**似然**：$L(p) = \prod_i p^{x_i}(1-p)^{1-x_i} = p^7 (1-p)^3$。让它最大的 $p$ 叫**最大似然估计**（MLE）。实践中总是取对数——乘积变求和，而且不会下溢：

In [ ]:
flips = np.array([1, 1, 0, 1, 1, 1, 0, 1, 0, 1])
ps = np.linspace(0.01, 0.99, 197)
log_lik = np.array([np.sum(flips * np.log(p) + (1 - flips) * np.log(1 - p)) for p in ps])
print(f"对数似然最大的 p = {ps[np.argmax(log_lik)]:.2f}（样本均值 {flips.mean():.2f}）")

many = rng.binomial(1, 0.7, size=2000)
probs = np.where(many == 1, 0.7, 0.3)
print("2000 次抛硬币：似然直接连乘 =", np.prod(probs), "← 下溢成 0 | 对数似然 =", round(np.sum(np.log(probs)), 1))

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].plot(ps, np.exp(log_lik))
axes[0].set_title("Likelihood p^7 (1-p)^3")
axes[1].plot(ps, log_lik)
axes[1].set_title("Log-likelihood: same maximiser, p = 0.7")
for ax in axes:
    ax.axvline(0.7, color="gray", ls="--")
    ax.set_xlabel("p")
plt.tight_layout()
plt.show()

**交叉熵就是负对数似然**。分类模型给每个样本输出一个类别分布 $\hat\pi = \mathrm{softmax}(z)$；整个数据集在这个模型下的对数似然是 $\sum_i \log \hat\pi_{i, y_i}$。取负号、除以样本数——这正是你从第 5 周用到现在的交叉熵损失。所以**最小化交叉熵 = 最大化似然**（d2l 附录 22.11.5.3 的推导结论是 minimizing CE is equivalent to maximizing the log-likelihood function；22.11.6 小结的原话："Minimizing cross-entropy loss is equivalent to maximizing the log-likelihood function"）。

两分类、以及下周 VAE 的重建项用的是伯努利版本：$\mathrm{BCE}(\hat p, x) = -[x \log \hat p + (1-x)\log(1-\hat p)]$。数值验证两者：

In [ ]:
torch.manual_seed(0)
logits = torch.randn(6, 10)                         # 6 个样本、10 类
labels = torch.randint(0, 10, (6,))
nll = -F.log_softmax(logits, dim=1)[torch.arange(6), labels].mean()
print(f"F.cross_entropy = {F.cross_entropy(logits, labels).item():.6f} | 手算负对数似然 = {nll.item():.6f}")

p_hat = torch.rand(8)                               # 模型给出的"像素为 1"的概率
x = torch.rand(8)                                   # 目标：可以是 [0, 1] 之间的灰度（下周的 MNIST 像素）
bern_nll = -(x * torch.log(p_hat) + (1 - x) * torch.log(1 - p_hat)).mean()
print(f"F.binary_cross_entropy = {F.binary_cross_entropy(p_hat, x).item():.6f} | 伯努利负对数似然 = {bern_nll.item():.6f}")

### 4. 熵：一个分布有多"不确定"

$$H(p) = -\sum_x p(x) \log p(x) = \mathbb{E}_p[-\log p(x)]$$

确定的分布熵为 0；$K$ 个类的均匀分布熵最大，等于 $\log K$。你见过这个数：第 9 周体检 ① 的"初始损失 ≈ ln 10"，就是均匀预测下 10 类交叉熵的值。伯努利的熵在 $p = 0.5$ 时最大（$\ln 2$），和 S1 里 DistilBERT 的初始损失 0.693 是同一个数：

In [ ]:
def entropy(probs):
    probs = np.asarray(probs, dtype=float)
    probs = probs[probs > 0]                        # 约定 0·log 0 = 0
    return np.sum(probs * np.log(1 / probs))        # 即 −Σ p log p


print(f"均匀 10 类: {entropy(np.ones(10) / 10):.4f}（ln 10 = {np.log(10):.4f}）| 确定分布: {entropy([1, 0, 0]):.4f}"
      f" | 公平硬币: {entropy([0.5, 0.5]):.4f}（ln 2 = {np.log(2):.4f}）")

# 高斯的熵有闭式解 ½ log(2πeσ²)；它也是一个期望 E[−log p(x)]，可以用蒙特卡洛验证
sigma = 2.0
samples = rng.normal(0, sigma, size=200_000)
mc_entropy = np.mean(-np.log(gauss_pdf(samples, 0, sigma)))
print(f"N(0, 2²) 的熵：闭式 {0.5 * np.log(2 * np.pi * np.e * sigma ** 2):.4f} | 蒙特卡洛 {mc_entropy:.4f}")

pb = np.linspace(0.001, 0.999, 200)
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(pb, [entropy([p, 1 - p]) for p in pb])
ax.axhline(np.log(2), color="gray", ls="--", label="ln 2")
ax.set_xlabel("p")
ax.set_ylabel("Entropy (nats)")
ax.set_title("A coin is most uncertain at p = 0.5")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### 5. KL 散度：两个分布差多远

$$\mathrm{KL}(p \,\|\, q) = \sum_x p(x) \log\frac{p(x)}{q(x)} = \mathbb{E}_{p}\big[\log p(x) - \log q(x)\big]$$

三个性质：

1. $\mathrm{KL}(p\|q) \ge 0$，当且仅当 $p = q$ 时为 0；
2. **不对称**：$\mathrm{KL}(p\|q) \ne \mathrm{KL}(q\|p)$——所以它不是"距离"，是"散度"；
3. **交叉熵 = 熵 + KL**：$H(p, q) = -\mathbb{E}_p[\log q] = H(p) + \mathrm{KL}(p\|q)$。训练分类器时 $p$ 是数据（固定），所以最小化交叉熵就是最小化 $\mathrm{KL}(p_{\text{data}} \| q_{\text{model}})$——把模型分布推向数据分布。

先在离散分布上验证这三条：

In [ ]:
p = np.array([0.1, 0.2, 0.3, 0.4])
q = np.ones(4) / 4                                   # 均匀分布


def kl_discrete(p, q):
    return np.sum(p * np.log(p / q))


cross_entropy = -np.sum(p * np.log(q))
print(f"KL(p‖q) = {kl_discrete(p, q):.4f} | KL(q‖p) = {kl_discrete(q, p):.4f} | KL(p‖p) = {kl_discrete(p, p):.4f}")
print(f"交叉熵 H(p, q) = {cross_entropy:.4f} | 熵 H(p) + KL(p‖q) = {entropy(p) + kl_discrete(p, q):.4f}")
print(f"q 是均匀分布时还有一个小结论：KL(p‖q) = log K − H(p) = {np.log(4) - entropy(p):.4f}")

两个方向的 KL 不一样（0.106 vs 0.122）。极端情况下差得更多：如果 $p$ 在某个位置是 0 而 $q$ 不是，$\mathrm{KL}(q\|p)$ 就是无穷大。

### 6. 两个一维高斯的 KL：解析解 vs 蒙特卡洛

连续分布把求和换成积分。两个一维高斯之间的 KL 有闭式解：

$$\mathrm{KL}\big(\mathcal{N}(\mu_1, \sigma_1^2) \,\|\, \mathcal{N}(\mu_2, \sigma_2^2)\big) = \log\frac{\sigma_2}{\sigma_1} + \frac{\sigma_1^2 + (\mu_1 - \mu_2)^2}{2\sigma_2^2} - \frac{1}{2}$$

KL 是一个期望，所以也可以**用蒙特卡洛估计**：从 $p$ 抽样 $x_i$，对 $\log p(x_i) - \log q(x_i)$ 求平均。两种算法应该给出同一个数——这就是"用代码验证公式"：

In [ ]:
def gaussian_kl(mu1, sigma1, mu2, sigma2):
    return math.log(sigma2 / sigma1) + (sigma1 ** 2 + (mu1 - mu2) ** 2) / (2 * sigma2 ** 2) - 0.5


def log_gauss(x, mu, sigma):
    # 高斯的对数密度：log p(x) = −½ log(2πσ²) − (x − μ)² / (2σ²)
    return -0.5 * np.log(2 * np.pi * sigma ** 2) - (x - mu) ** 2 / (2 * sigma ** 2)


def mc_gaussian_kl(mu1, sigma1, mu2, sigma2, n, rng):
    x = rng.normal(mu1, sigma1, size=n)                 # 从 p 抽样
    return np.mean(log_gauss(x, mu1, sigma1) - log_gauss(x, mu2, sigma2))


exact = gaussian_kl(1.0, 0.5, 0.0, 1.0)
estimate = mc_gaussian_kl(1.0, 0.5, 0.0, 1.0, 100_000, np.random.default_rng(0))
print(f"KL(N(1, 0.5²) ‖ N(0, 1))：解析 {exact:.4f} | 蒙特卡洛（10 万个样本）{estimate:.4f}")
print(f"反过来 KL(N(0, 1) ‖ N(1, 0.5²))：解析 {gaussian_kl(0.0, 1.0, 1.0, 0.5):.4f}  ← 不对称")
ck.gaussian_kl(gaussian_kl)
ck.mc_kl(estimate, exact)

In [ ]:
ns = np.unique(np.logspace(1, 5, 30).astype(int))
gen = np.random.default_rng(1)
running = [mc_gaussian_kl(1.0, 0.5, 0.0, 1.0, n_samples, gen) for n_samples in ns]
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.semilogx(ns, running, "o-", ms=3, label="Monte Carlo estimate")
ax.axhline(exact, color="red", ls="--", label=f"Closed form {exact:.4f}")
ax.set_xlabel("Number of samples from p")
ax.set_ylabel("KL(p || q)")
ax.set_title("The Monte Carlo estimate converges to the closed-form KL")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### 7. VAE 会用到的那一项：$\mathrm{KL}(\mathcal{N}(\mu, \sigma^2)\,\|\,\mathcal{N}(0, 1))$

下周的 VAE 里，编码器为每张图输出一个高斯 $\mathcal{N}(\mu, \sigma^2)$，损失里有一项要求它别离标准正态 $\mathcal{N}(0, 1)$ 太远。把上面的公式代入 $\mu_2 = 0, \sigma_2 = 1$：

$$\mathrm{KL} = -\log\sigma + \frac{\sigma^2 + \mu^2}{2} - \frac{1}{2} = -\frac{1}{2}\left(1 + \log\sigma^2 - \mu^2 - \sigma^2\right)$$

潜变量有 $d$ 维、各维独立时，KL 是各维之和：$-\frac{1}{2}\sum_{j=1}^{d}(1 + \log\sigma_j^2 - \mu_j^2 - \sigma_j^2)$——这正是 Kingma & Welling 2013 附录 B 的结果，也是 pytorch/examples 里 VAE 代码的那一行。编码器输出的通常是 $\log\sigma^2$（记作 `logvar`）而不是 $\sigma$：它可以取任意实数，不用担心 $\sigma$ 变成负数。

In [ ]:
mu = torch.tensor([0.0, 1.0, -0.5])
logvar = torch.tensor([0.0, -1.0, 0.8])
kl_vae = -0.5 * torch.sum(1 + logvar - mu ** 2 - logvar.exp())
kl_from_general = sum(gaussian_kl(m.item(), math.exp(0.5 * lv.item()), 0.0, 1.0) for m, lv in zip(mu, logvar))
print(f"VAE 的 KL 公式 {kl_vae.item():.6f} | 通用公式逐维相加 {kl_from_general:.6f}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
grid = np.linspace(-3, 3, 200)
axes[0].plot(grid, [gaussian_kl(m, 1.0, 0, 1) for m in grid])
axes[0].set_xlabel("mu (sigma = 1)")
axes[0].set_title("KL grows as mu² / 2")
sig = np.linspace(0.1, 3, 200)
axes[1].plot(sig, [gaussian_kl(0, s, 0, 1) for s in sig])
axes[1].set_xlabel("sigma (mu = 0)")
axes[1].set_title("Too narrow or too wide both cost KL")
for ax in axes:
    ax.set_ylabel("KL to N(0, 1)")
    ax.axvline(0 if ax is axes[0] else 1, color="gray", ls="--")
plt.tight_layout()
plt.show()

两张图读出 KL 项在 VAE 里的作用：它把每张图的编码**往原点拉**（$\mu \to 0$），并且**不许分布缩成一个点**（$\sigma \to 0$ 时 $-\log\sigma \to \infty$）。下周你会看到，正是这一项让潜空间变得"连续、可采样"。

---
## S4 · 复现进度自查

第 13 周你选定了一篇论文、一张图或一个表，写了复现计划。现在停下来，用 **5 分钟**按下面的清单把进度写下来（或讲给自己、学习伙伴听），再对照后面的坑清单排错。

### 5 分钟进度清单

1. **目标**（30 秒）：复现原文的哪张图 / 哪个表；验收口径是"数值"还是"趋势"；
2. **已复现的部分**（2 分钟）：展示你的图或表，**和原文并排**；至少说出一个具体数字的对比（原文多少、你多少）；
3. **与原文的差异**（1 分钟）：哪些设置你改了（为了算力或时间）、哪些是原文没写清楚、你自己猜的；
4. **卡点**（1 分钟）：卡在哪里、已经试过什么、下一步打算试什么；
5. **写复现报告前的计划**（30 秒）：还差哪些种子、哪条曲线。

### 常见复现坑清单（对照检查一遍）

| 坑 | 典型表现 | 先查什么 |
| --- | --- | --- |
| **数据版本** | 数据集大小、类别数和论文对不上 | 训练/测试集张数；是否用了论文的标准划分；是否混入了验证集 |
| **预处理** | 准确率整体偏低几个点 | 归一化用的均值方差；输入尺寸；数据增强（随机裁剪、翻转）开没开 |
| **学习率与调度** | 曲线形状不对：收敛太慢或后期不降 | 初始学习率；有没有衰减（ResNet 在 32k/48k 步除以 10）；warmup |
| **epoch / 步数** | 你的曲线比原文"短" | 原文按 epoch 还是按迭代步数计；缩短训练后只能比较趋势 |
| **随机性** | 不同运行结果差很多 | 种子个数（≥3）；原文报的是单次运行还是均值；你的差异是否小于种子间标准差 |
| **评估口径** | 数字差得离谱或方向反了 | 原文画的是 training cost 还是 test error（Adam 图 2a 是**训练**损失；ResNet 图 6 的虚线是**训练**误差）；误差 = 1 − 准确率 |
| **模型细节** | 参数量对不上 | 每层宽度；Dropout 是否也作用于输入层（Dropout 论文输入层保留率 0.8）；BatchNorm 位置 |

下周 S4 写正式的复现报告，按 `templates/replication_report.md`。

### 本周小结

- 预训练模型要用**它自己的分词器**（子词、`[CLS]/[SEP]`、`attention_mask`）；`AutoModelForSequenceClassification` = 预训练身体 + 随机初始化的分类头（加载报告里的 MISSING）
- 微调就是第 8 周的训练循环：AdamW、小学习率（2e-5）；CPU 缩小配置至少要 3 个 epoch，1 个 epoch 是否学得动取决于种子
- 同样只用 500 条训练时，微调比 TF-IDF 高几个百分点（本机单种子 0.842 vs 0.780，另 3 个种子 0.798–0.824；作业用 3 个种子确认）；赢过 5000 条训练的 TF-IDF 要在 Colab T4 上用大配置（5000 条 / 256 token）
- 可训练参数的光谱：线性探针（≈0.59M）→ 冻结底部若干层 → 全量微调（67M）；LoRA 只训练低秩增量 $\frac{\alpha}{r}BA$（$A$ 高斯、$B$ 零 → 起点不变），r=4 时 A/B 只有 73,728 个
- PEFT 的 `print_trainable_parameters` 把分类头也算进去了：数参数要拆账
- 概率工具箱：蒙特卡洛估期望（误差 ∝ $1/\sqrt{n}$）；交叉熵 = 负对数似然；熵；KL ≥ 0、不对称、交叉熵 = 熵 + KL；高斯 KL 的闭式解与 VAE 用的 $-\frac{1}{2}\sum(1 + \log\sigma^2 - \mu^2 - \sigma^2)$